# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a walkthrough for loading, exploring, and analyzing the FAIR² dataset, describing protein abundance and sequence analysis from extracellular vesicles, using the `mlcroissant` library in Python.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Print a summary description
print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Examine the available record sets, their fields, and associated `@id`s in the dataset.

Record sets define the major tables or logical sets of records in the dataset. Fields define the columns in each record set.

Let's enumerate all record sets, fields, and columns by their `@id` as per the Croissant schema.

In [ ]:
# List all available record sets
record_sets = list(dataset.metadata.record_sets)
print("Available Record Sets and their @id:")
for rs in record_sets:
    print(f"  Name: {rs.name}\n     @id: {rs.id}")
    print(f"     Description: {rs.description if hasattr(rs, 'description') else ''}")
    # List all fields for this record set
    if hasattr(rs, 'fields'):
        print("     Fields/Columns (@id):")
        for field in rs.fields:
            print(f"        {field.name} (@id: {field.id}) | Type: {field.data_type}")
    print("")

## Example Record Preview
Print a small sample of records from a record set to inspect its structure.

*This example uses the first record set detected. Adjust the `record_set_id` variable below to the appropriate `@id` as needed for other tables.*

In [ ]:
# Preview a few records from the first record set
if len(record_sets) > 0:
    first_record_set = record_sets[0]
    print(f"Previewing records for record set: {first_record_set.name} (@id: {first_record_set.id})\n")
    try:
        for i, rec in enumerate(dataset.records(record_set=first_record_set.id)):
            print(rec)
            if i >= 2:
                break
    except Exception as e:
        print(f"Could not read records: {e}")
else:
    print("No record sets found in the metadata.")

## 3. Data Extraction
Load the data from **each record set** into pandas DataFrames for further analysis.

All references use the `@id` values discovered above. This ensures robust data extraction and alignment with the dataset schema.

In [ ]:
# Extract records from each record set into pandas DataFrames
dataframes = {}
for rs in record_sets:
    print(f"Loading records for: {rs.name} (@id: {rs.id}) ...")
    try:
        df = pd.DataFrame(dataset.records(record_set=rs.id))
        dataframes[rs.id] = df
        print(f"  Loaded {len(df)} records. Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Failed to load records for record set {rs.id}: {e}")

# Select one record set and show its first records
if len(record_sets) > 0:
    selected_id = record_sets[0].id
    print(f"\nColumns in selected DataFrame ({selected_id}):\n{dataframes[selected_id].columns.tolist()}")
    display(dataframes[selected_id].head())

## 4. Exploratory Data Analysis (EDA)

Let's explore and process the dataset by filtering and normalizing numeric fields, and grouping by categories, all using field `@id` references.

*Adjust field and group names to your task; here, we auto-detect the first available numeric field and a possible group-by field.*

In [ ]:
import numpy as np

selected_rs = record_sets[0]
df = dataframes[selected_rs.id]

# Try to auto-detect numeric fields (float/int)
numeric_fields = [field for field in selected_rs.fields if field.data_type in ['Float', 'Integer', 'Number']]

if numeric_fields:
    numeric_field_id = numeric_fields[0].id  # e.g., 'coverage_percentage', or similar
    print(f"Using numeric field: {numeric_field_id}")
else:
    print("No numeric field found, please set 'numeric_field_id' manually from previous overview.")
    numeric_field_id = None

# Filter out records where > threshold
threshold = 10
if numeric_field_id and numeric_field_id in df.columns and np.issubdtype(df[numeric_field_id].dtype, np.number):
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} records")

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()

    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
else:
    print(f"Numeric field '{numeric_field_id}' is not available in the selected DataFrame or not numeric.")

# Attempt to auto-select a group field (categorical)
categorical_fields = [field for field in selected_rs.fields if field.data_type in ['Text', 'String']]
group_field_id = categorical_fields[0].id if categorical_fields else None

if group_field_id and group_field_id in df.columns and numeric_field_id and numeric_field_id in df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
    print(grouped_df.head())
else:
    print("No suitable group field available for grouping.")

## 5. Visualization
Visualize the distribution of the selected numeric field and the grouped means, if appropriate.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram for the filtered numeric field
if numeric_field_id and numeric_field_id in df.columns and np.issubdtype(df[numeric_field_id].dtype, np.number):
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id} (>{threshold})")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

# Plot grouped means if available
if group_field_id and group_field_id in df.columns and numeric_field_id and numeric_field_id in df.columns:
    means = filtered_df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False).head(10)
    plt.figure(figsize=(10, 4))
    sns.barplot(x=means.values, y=means.index, palette="Blues_d")
    plt.title(f"Top 10 {group_field_id} by average {numeric_field_id}")
    plt.xlabel(f"Average {numeric_field_id}")
    plt.ylabel(group_field_id)
    plt.tight_layout()
    plt.show()
else:
    print("No grouped field available for visualization.")

## 6. Conclusion
In this notebook, we've loaded and explored the FAIR² mass spectrometry dataset using `mlcroissant` by referencing all schema entities through their `@id` identifiers. This enables reproducible and robust data access across different future releases.

- We listed all record sets and fields, loaded data to pandas DataFrames, and previewed the structure.
- Basic filtering/normalization/grouping/visualization steps were demonstrated for numeric and categorical fields.

Feel free to further adjust field selections and analysis to fit your scientific questions.